In [31]:
"""1.This cell contains the code loading,checking and reshaping the dataset for further processing""" 

import torch 
from torchvision import transforms,datasets
from torch.utils.data import DataLoader,random_split




device = torch.device("cuda" if torch.cuda.is_available() else "cpu") #Fallback to CPU if GPU is not available
if(device.type == "cuda"):
    print("GPU is available")
    print("Converting Arrays to Tensors . . . .")


categories = ["bellflower","daisy","dandelion","lotus","rose","sunflower","tulip"] #Categories of flowers in the dataset

transform = transforms.Compose([transforms.ToTensor(),transforms.Resize((150,150))]) #Converting images to tensors
dataset = datasets.ImageFolder(root="flowers", transform = transform)


#Spliting the dataset into training and testing datasets
train_dataset, test_dataset = random_split(dataset, [int(len(dataset)*0.8), int(len(dataset)*0.2)]) #Splitting the dataset into training and testing datasets


train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True) #Creating a dataloader to load the dataset to GPU in batches

test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=True) #Creating a dataloader to load the dataset in 

GPU is available
Converting Arrays to Tensors . . . .


In [32]:
from torch import nn
import torch.optim as optim

class FlowerClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        #Convolutional Layer1 forward pass
        self.cnn_layer = nn.Sequential(nn.Conv2d(in_channels=3, out_channels =32,kernel_size=3,padding=1,stride = 2,bias = True) ,
                                       nn.BatchNorm2d(32),
                                       #Activation Layer
                                       nn.ReLU(),

                                       #Pooling Layer
                                       nn.MaxPool2d(kernel_size=3,stride =1,padding=1))





        self.cnn_layer_2 = nn.Sequential(nn.Conv2d(in_channels=32, out_channels =64,kernel_size=3,padding=1,stride = 2,bias = True),
                    nn.BatchNorm2d(64),
                    nn.ReLU(),
                    nn.MaxPool2d(kernel_size=3,stride =1,padding=1))

        self.cnn_layer_3 = nn.Sequential(nn.Conv2d(in_channels=64, out_channels =128,kernel_size=3,padding=1,stride = 2,bias = True),
                    nn.BatchNorm2d(128),
                    nn.ReLU(),
                    nn.MaxPool2d(kernel_size=3,stride =1,padding=1))

    

        #Neural Network forward pass
        self.flatten = nn.Flatten()


        self.layer_1 = nn.Sequential(nn.Linear(in_features = 128*19*19,out_features =128,bias=True),
                                     nn.ReLU())
        self.layer_2 = nn.Sequential(nn.Linear(in_features = 128,out_features =64,bias=True),
                                             nn.ReLU())
        self.layer_3 = nn.Sequential(nn.Linear(in_features = 64,out_features =32,bias=True),
                                             nn.ReLU())

        #Output Layer
        self.out = nn.Linear(in_features = 32, out_features=7)

    

    def forward(self,x):

        x = self.cnn_layer(x)
        x = self.cnn_layer_2(x)
        x = self.cnn_layer_3(x)
        x = self.flatten(x)
        x = (self.layer_1(x))
        x = (self.layer_2(x))
        x = (self.layer_3(x))
        x = self.out(x)
        return x

model = FlowerClassifier().to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
print(model)

FlowerClassifier(
  (cnn_layer): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=3, stride=1, padding=1, dilation=1, ceil_mode=False)
  )
  (cnn_layer_2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=3, stride=1, padding=1, dilation=1, ceil_mode=False)
  )
  (cnn_layer_3): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=3, stride=1, padding=1, dilation=1, ceil_mode=False)
  )
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (layer_1): 

In [33]:

epochs = 20

for epoch in range(epochs):

    model.train()
    

    running_train_loss = 0.0
    correct_train = 0
    total_train = 0

    for images, labels in train_dataloader:
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass & loss
        output = model(images)
        loss = loss_function(output, labels)

        # Optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * images.size(0)
        _, preds = torch.max(output, 1)
        correct_train += (preds == labels).sum().item()
        total_train += labels.size(0)

    # Step 3: Now both variables exist and can be safely divided
    train_loss = running_train_loss / total_train
    train_acc = (correct_train / total_train) * 100


    model.eval()
    running_test_loss = 0.0
    correct_test = 0
    total_test = 0

    with torch.inference_mode():
        for images, labels in test_dataloader:
            images = images.to(device)
            labels = labels.to(device)

            test_output = model(images)
            t_loss = loss_function(test_output, labels)

            running_test_loss += t_loss.item() * images.size(0)
            _, test_preds = torch.max(test_output, 1)
            correct_test += (test_preds == labels).sum().item()
            total_test += labels.size(0)

    test_loss = running_test_loss / total_test
    test_acc = (correct_test / total_test) * 100
    
    print(f"Epoch [{epoch+1:02d}/{epochs:02d}] | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
          f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%")

Epoch [01/20] | Train Loss: 1.2638 | Train Acc: 53.58% | Test Loss: 1.0091 | Test Acc: 64.73%
Epoch [02/20] | Train Loss: 0.9040 | Train Acc: 68.37% | Test Loss: 0.9409 | Test Acc: 66.07%
Epoch [03/20] | Train Loss: 0.7948 | Train Acc: 71.84% | Test Loss: 0.8324 | Test Acc: 70.13%
Epoch [04/20] | Train Loss: 0.6940 | Train Acc: 75.94% | Test Loss: 0.7780 | Test Acc: 72.63%
Epoch [05/20] | Train Loss: 0.6242 | Train Acc: 78.07% | Test Loss: 0.7188 | Test Acc: 74.42%
Epoch [06/20] | Train Loss: 0.5673 | Train Acc: 80.42% | Test Loss: 0.6235 | Test Acc: 78.44%
Epoch [07/20] | Train Loss: 0.4926 | Train Acc: 82.97% | Test Loss: 0.6733 | Test Acc: 77.14%
Epoch [08/20] | Train Loss: 0.4409 | Train Acc: 85.02% | Test Loss: 0.6891 | Test Acc: 76.88%
Epoch [09/20] | Train Loss: 0.3985 | Train Acc: 86.43% | Test Loss: 0.6992 | Test Acc: 77.46%
Epoch [10/20] | Train Loss: 0.3698 | Train Acc: 86.79% | Test Loss: 0.5551 | Test Acc: 82.28%
Epoch [11/20] | Train Loss: 0.3397 | Train Acc: 88.47% | Tes